# Overture — multi-theme download (live)

A single request can pull several themes / types at once; one file is written per feature type. Here we grab places and building footprints for the same block, targeting the release Overture publishes right now.

## Setup

Pull in the pieces we need: the `EarthLens` entry point to run the download, `releases` to see which Overture release the download will target, and `geopandas` to inspect the written GeoParquet files.

In [ ]:
import geopandas as gpd

from earthlens.core import EarthLens
from earthlens.overture import releases

## Area of interest

Define a small Times Square block as a `[south, north]` / `[west, east]` bounding box, and the directory the output files will be written to.

In [ ]:
LAT_LIM = [40.757, 40.759]  # [south, north]
LON_LIM = [-73.987, -73.984]  # [west, east] — a Times Square block
OUT = "_overture_out"

## Which release will this read?

Overture publishes a dated release roughly monthly and keeps only the newest one or two on S3, pruning the rest — so a release id is a moving target, and an id that worked last quarter may no longer exist. Leaving `release=None` (the default) targets whatever is published now, which is what we want here.

You *can* pin a release when you need a byte-reproducible download, but the pin holds only while that release is still on S3; once it is pruned the fetch fails with `No files found that match the pattern`. Ask Overture what it is serving today:

In [ ]:
release = releases.latest_release()
release

## Download both themes

Request `places` and `buildings` in one call. Overture writes one GeoParquet file per feature type, so `download()` returns a list of paths — one per type.

In [ ]:
engine = EarthLens(
    data_source="overture",
    variables={"places": [], "buildings": []},
    aoi=[LON_LIM[0], LAT_LIM[0], LON_LIM[1], LAT_LIM[1]],
    path=OUT,
)
paths = engine.download()
[p.name for p in paths]

## Inspect the results

Read each written file back with `geopandas` and report its feature count and the set of per-row license ids (buildings are OSM-derived ODbL-1.0; places carry permissive licenses).

In [ ]:
for p in paths:
    gdf = gpd.read_parquet(p)
    print(f'{p.name}: {len(gdf)} features, licenses={sorted(gdf.license_id.unique())}')